# PhysicsFormer Ablation Study

**Copyright (c) 2026 Style Machine LLC. All rights reserved.**

**Author:** Jesse Pokora

---

## Purpose

Determine the contribution of each physics-specific feature to model performance with **statistical significance**.

## Ablation Conditions

| Condition | `use_physics_bias` | `use_energy_conservation` | `use_graph_attention` | `use_learned_scaling` |
|-----------|-------------------|--------------------------|----------------------|----------------------|
| **Full Model** | ✓ | ✓ | ✓ | ✓ |
| **No Physics Bias** | ✗ | ✓ | ✓ | ✓ |
| **No Energy Conservation** | ✓ | ✗ | ✓ | ✓ |
| **No Graph Attention** | ✓ | ✓ | ✗ | ✓ |
| **No Learned Scaling** | ✓ | ✓ | ✓ | ✗ |
| **Vanilla Transformer** | ✗ | ✗ | ✗ | ✗ |

## Statistical Analysis

- **Multiple runs** (N=5) per condition for variance estimation
- **95% Confidence Intervals** for all metrics
- **Paired t-tests** comparing each ablation to full model
- **Effect size** (Cohen's d) for practical significance

## Metrics

- **Trajectory MSE** - State prediction error
- **Schema Classification Accuracy** - Physics scenario identification
- **Energy Conservation Error** - Physics consistency
- **Convergence Speed** - Epochs to reach target loss

In [ ]:
# ============================================================
# CELL 1: IMPORTS AND CONFIGURATION
# ============================================================

print("="*70)
print("PHYSICS FORMER ABLATION STUDY")
print("="*70)
print("Determining contribution of each physics feature with statistical significance")
print()

import os
import sys
import json
import math
import copy
import time
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple
from enum import Enum

print("✓ Standard library imports loaded")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import numpy as np
from scipy import stats

print("✓ PyTorch, NumPy, SciPy loaded")

# Mount Google Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')
ON_COLAB = True
GDRIVE_DATA = "/content/drive/MyDrive/physics_action_predictor/data"
RESULTS_DIR = "/content/drive/MyDrive/physics_action_predictor/ablation_results"

print("✓ Google Drive mounted")

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for ablation study. No GPU detected.")

print()
print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"  Device: {device}")
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  Results directory: {RESULTS_DIR}")

# Ablation configuration
NUM_RUNS = 5  # Runs per condition for statistical significance
MAX_EPOCHS = 50  # Max epochs per run
EARLY_STOP_PATIENCE = 10
BATCH_SIZE = 32
SEED_BASE = 42  # Seeds will be 42, 43, 44, 45, 46

print()
print(f"  Runs per condition: {NUM_RUNS}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Seed base: {SEED_BASE}")
print("="*70)

In [ ]:
# ============================================================
# CELL 2: ABLATION CONDITIONS
# ============================================================

# All available ablation conditions (for reference and future use)
ALL_ABLATION_CONDITIONS = {
    'full_model': {
        'use_physics_bias': True,
        'use_energy_conservation': True,
        'use_graph_attention': True,
        'use_learned_scaling': True,
        'description': 'Full model with all physics features'
    },
    'no_physics_bias': {
        'use_physics_bias': False,
        'use_energy_conservation': True,
        'use_graph_attention': True,
        'use_learned_scaling': True,
        'description': 'Without physics-informed attention bias'
    },
    'no_energy_conservation': {
        'use_physics_bias': True,
        'use_energy_conservation': False,
        'use_graph_attention': True,
        'use_learned_scaling': True,
        'description': 'Without pHMARL energy conservation'
    },
    'no_graph_attention': {
        'use_physics_bias': True,
        'use_energy_conservation': True,
        'use_graph_attention': False,
        'use_learned_scaling': True,
        'description': 'Without Body Transformer graph attention'
    },
    'no_learned_scaling': {
        'use_physics_bias': True,
        'use_energy_conservation': True,
        'use_graph_attention': True,
        'use_learned_scaling': False,
        'description': 'Without per-feature learned scaling'
    },
    'vanilla_transformer': {
        'use_physics_bias': False,
        'use_energy_conservation': False,
        'use_graph_attention': False,
        'use_learned_scaling': False,
        'description': 'Vanilla transformer (no physics features)'
    }
}

# Conditions to run in this experiment (excluding graph attention for now)
ABLATION_CONDITIONS = {
    'full_model': ALL_ABLATION_CONDITIONS['full_model'],
    'no_physics_bias': ALL_ABLATION_CONDITIONS['no_physics_bias'],
    'no_energy_conservation': ALL_ABLATION_CONDITIONS['no_energy_conservation'],
    'no_learned_scaling': ALL_ABLATION_CONDITIONS['no_learned_scaling'],
    'vanilla_transformer': ALL_ABLATION_CONDITIONS['vanilla_transformer'],
    # 'no_graph_attention': ALL_ABLATION_CONDITIONS['no_graph_attention'],  # EXCLUDED for this run
}

print("Ablation Conditions for this experiment:")
print("=" * 70)
print(f"  Running {len(ABLATION_CONDITIONS)} conditions (graph attention excluded)")
print()
for name, config in ABLATION_CONDITIONS.items():
    print(f"  {name}:")
    print(f"    {config['description']}")
    print(f"    physics_bias={config['use_physics_bias']}, energy={config['use_energy_conservation']}, "
          f"graph={config['use_graph_attention']}, scaling={config['use_learned_scaling']}")
    print()

In [ ]:
# ============================================================
# CELL 3: STATISTICAL ANALYSIS FUNCTIONS
# ============================================================

def compute_confidence_interval(data: List[float], confidence: float = 0.95) -> Tuple[float, float, float]:
    """Compute mean and confidence interval.
    
    Returns: (mean, ci_lower, ci_upper)
    """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)  # Standard error of the mean
    
    # t-value for confidence level
    t_value = stats.t.ppf((1 + confidence) / 2, n - 1)
    margin = t_value * se
    
    return mean, mean - margin, mean + margin


def paired_t_test(baseline: List[float], ablation: List[float]) -> Dict:
    """Perform paired t-test comparing ablation to baseline.
    
    Returns dict with t-statistic, p-value, and significance.
    """
    t_stat, p_value = stats.ttest_rel(baseline, ablation)
    
    # Cohen's d for effect size
    diff = np.array(baseline) - np.array(ablation)
    cohens_d = np.mean(diff) / np.std(diff, ddof=1) if np.std(diff) > 0 else 0
    
    return {
        't_statistic': t_stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'significant_005': p_value < 0.05,
        'significant_001': p_value < 0.01,
        'effect_size': 'large' if abs(cohens_d) > 0.8 else 'medium' if abs(cohens_d) > 0.5 else 'small'
    }


def format_result(mean: float, ci_low: float, ci_high: float, precision: int = 4) -> str:
    """Format result as 'mean [ci_low, ci_high]'."""
    return f"{mean:.{precision}f} [{ci_low:.{precision}f}, {ci_high:.{precision}f}]"


def analyze_ablation_results(results: Dict[str, List[Dict]]) -> Dict:
    """Analyze ablation results with statistical tests.
    
    Args:
        results: Dict mapping condition name to list of run results
        
    Returns:
        Analysis dict with confidence intervals and significance tests
    """
    analysis = {}
    
    # Get baseline (full model) results
    baseline_mse = [r['final_mse'] for r in results['full_model']]
    baseline_acc = [r['schema_accuracy'] for r in results['full_model']]
    baseline_epochs = [r['epochs_to_converge'] for r in results['full_model']]
    
    for condition, runs in results.items():
        mse_values = [r['final_mse'] for r in runs]
        acc_values = [r['schema_accuracy'] for r in runs]
        epoch_values = [r['epochs_to_converge'] for r in runs]
        
        # Confidence intervals
        mse_mean, mse_ci_low, mse_ci_high = compute_confidence_interval(mse_values)
        acc_mean, acc_ci_low, acc_ci_high = compute_confidence_interval(acc_values)
        epoch_mean, epoch_ci_low, epoch_ci_high = compute_confidence_interval(epoch_values)
        
        analysis[condition] = {
            'mse': {
                'mean': mse_mean,
                'ci_95': (mse_ci_low, mse_ci_high),
                'std': np.std(mse_values),
                'formatted': format_result(mse_mean, mse_ci_low, mse_ci_high)
            },
            'accuracy': {
                'mean': acc_mean,
                'ci_95': (acc_ci_low, acc_ci_high),
                'std': np.std(acc_values),
                'formatted': format_result(acc_mean, acc_ci_low, acc_ci_high, precision=2)
            },
            'convergence': {
                'mean': epoch_mean,
                'ci_95': (epoch_ci_low, epoch_ci_high),
                'std': np.std(epoch_values),
                'formatted': format_result(epoch_mean, epoch_ci_low, epoch_ci_high, precision=1)
            }
        }
        
        # Statistical tests vs baseline (skip for full_model)
        if condition != 'full_model':
            analysis[condition]['vs_baseline'] = {
                'mse_test': paired_t_test(baseline_mse, mse_values),
                'accuracy_test': paired_t_test(baseline_acc, acc_values),
                'convergence_test': paired_t_test(baseline_epochs, epoch_values)
            }
    
    return analysis


print("Statistical analysis functions defined")

In [ ]:
# ============================================================
# CELL 4: PHYSICS CONFIG WITH ABLATION SUPPORT
# ============================================================

@dataclass
class AblationConfig:
    """Configuration for PhysicsFormer ablation experiments."""
    # Model architecture
    num_objects: int = 20
    state_dim: int = 28
    embed_dim: int = 256
    num_heads: int = 8
    num_layers: int = 6
    ff_dim: int = 1024
    dropout: float = 0.1
    max_seq_len: int = 128
    
    # Modern improvements (always on)
    use_rope: bool = True
    use_rmsnorm: bool = True
    use_swiglu: bool = True
    use_flash_attention: bool = True
    
    # Physics-specific features (ablation targets)
    use_physics_bias: bool = True      # Physics-informed attention bias
    use_energy_conservation: bool = True  # pHMARL energy conservation
    use_graph_attention: bool = True   # Body Transformer graph attention
    use_learned_scaling: bool = True   # Per-feature learned scaling
    
    # Additional settings
    use_masked_attention: bool = True
    graph_edge_type: str = "spatial"
    spatial_threshold: float = 2.0
    use_hadamard_attention: bool = False
    per_feature_attention: bool = True
    hamiltonian_weight: float = 0.05
    
    def apply_ablation(self, ablation_config: Dict) -> 'AblationConfig':
        """Create a new config with ablation settings applied."""
        new_config = copy.deepcopy(self)
        for key, value in ablation_config.items():
            if key != 'description' and hasattr(new_config, key):
                setattr(new_config, key, value)
        return new_config


base_config = AblationConfig()
print(f"Base config: physics_bias={base_config.use_physics_bias}, "
      f"energy={base_config.use_energy_conservation}, "
      f"graph={base_config.use_graph_attention}, "
      f"scaling={base_config.use_learned_scaling}")

In [ ]:
# ============================================================
# CELL 4B: PHYSICSFORMER V2 MODEL WITH ABLATION SUPPORT
# ============================================================

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)


class SwiGLU(nn.Module):
    """SwiGLU activation function."""
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.w1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, out_dim, bias=False)
        self.w3 = nn.Linear(in_dim, hidden_dim, bias=False)
    
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class PhysicsBiasedAttention(nn.Module):
    """Attention with physics-informed bias terms."""
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1,
                 use_physics_bias: bool = True):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.use_physics_bias = use_physics_bias
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
        if use_physics_bias:
            # Physics bias network: distance, relative velocity, etc. -> bias
            self.physics_bias_net = nn.Sequential(
                nn.Linear(12, 64),  # 3 pos diff + 3 vel diff + 3 rel pos + 3 rel vel
                nn.ReLU(),
                nn.Linear(64, num_heads),
                nn.Tanh()
            )
    
    def forward(self, x, physics_features=None):
        B, T, N, D = x.shape
        x_flat = x.view(B * T, N, D)
        
        Q = self.q_proj(x_flat).view(B * T, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x_flat).view(B * T, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x_flat).view(B * T, N, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Add physics bias if enabled
        if self.use_physics_bias and physics_features is not None:
            # physics_features: [B*T, N, N, 12]
            bias = self.physics_bias_net(physics_features)  # [B*T, N, N, num_heads]
            bias = bias.permute(0, 3, 1, 2)  # [B*T, num_heads, N, N]
            scores = scores + bias
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B * T, N, D)
        out = self.out_proj(out)
        
        return out.view(B, T, N, D)


class PhysicsFormerLayer(nn.Module):
    """Single PhysicsFormer layer with ablation support."""
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, dropout: float,
                 use_physics_bias: bool = True, use_graph_attention: bool = True):
        super().__init__()
        self.use_graph_attention = use_graph_attention
        
        self.norm1 = RMSNorm(embed_dim)
        self.attn = PhysicsBiasedAttention(embed_dim, num_heads, dropout, use_physics_bias)
        self.norm2 = RMSNorm(embed_dim)
        self.ff = SwiGLU(embed_dim, ff_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
        if use_graph_attention:
            # Graph attention for object interactions
            self.graph_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout, batch_first=True)
            self.graph_norm = RMSNorm(embed_dim)
    
    def forward(self, x, physics_features=None, adjacency=None):
        # Self-attention with physics bias
        x = x + self.dropout(self.attn(self.norm1(x), physics_features))
        
        # Graph attention (if enabled)
        if self.use_graph_attention and adjacency is not None:
            B, T, N, D = x.shape
            x_flat = x.view(B * T, N, D)
            
            # Use adjacency as attention mask
            attn_mask = ~adjacency.view(B * T, N, N).bool() if adjacency is not None else None
            graph_out, _ = self.graph_attn(x_flat, x_flat, x_flat, attn_mask=attn_mask)
            x = x + self.dropout(graph_out.view(B, T, N, D))
        
        # Feedforward
        x = x + self.dropout(self.ff(self.norm2(x)))
        
        return x


class PhysicsFormerV2(nn.Module):
    """PhysicsFormer V2 with full ablation support."""
    def __init__(self, num_objects: int = 20, state_dim: int = 28, embed_dim: int = 256,
                 num_heads: int = 8, num_layers: int = 6, ff_dim: int = 1024,
                 dropout: float = 0.1, use_physics_bias: bool = True,
                 use_energy_conservation: bool = True, use_graph_attention: bool = True,
                 use_learned_scaling: bool = True, num_schemas: int = 37):
        super().__init__()
        
        self.num_objects = num_objects
        self.state_dim = state_dim
        self.embed_dim = embed_dim
        self.use_physics_bias = use_physics_bias
        self.use_energy_conservation = use_energy_conservation
        self.use_graph_attention = use_graph_attention
        self.use_learned_scaling = use_learned_scaling
        
        # Input embedding with optional learned scaling
        if use_learned_scaling:
            self.feature_scale = nn.Parameter(torch.ones(state_dim))
        self.input_proj = nn.Linear(state_dim, embed_dim)
        
        # Transformer layers
        self.layers = nn.ModuleList([
            PhysicsFormerLayer(embed_dim, num_heads, ff_dim, dropout,
                              use_physics_bias, use_graph_attention)
            for _ in range(num_layers)
        ])
        
        # Output projection
        self.output_norm = RMSNorm(embed_dim)
        self.output_proj = nn.Linear(embed_dim, state_dim)
        
        # Schema classifier
        self.schema_classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Linear(embed_dim // 2, num_schemas)
        )
        
        # Energy conservation components (pHMARL-inspired)
        if use_energy_conservation:
            self.energy_predictor = nn.Sequential(
                nn.Linear(embed_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 1)  # Predict total energy
            )
    
    def _compute_physics_features(self, x):
        """Compute pairwise physics features for attention bias."""
        if not self.use_physics_bias:
            return None
        
        B, T, N, D = x.shape
        
        # Extract position (0:3) and velocity (3:6)
        pos = x[:, :, :, :3]  # [B, T, N, 3]
        vel = x[:, :, :, 3:6] if D > 3 else torch.zeros_like(pos)
        
        # Compute pairwise features
        pos_i = pos.unsqueeze(3)  # [B, T, N, 1, 3]
        pos_j = pos.unsqueeze(2)  # [B, T, 1, N, 3]
        vel_i = vel.unsqueeze(3)
        vel_j = vel.unsqueeze(2)
        
        pos_diff = pos_i - pos_j  # [B, T, N, N, 3]
        vel_diff = vel_i - vel_j
        
        # Concatenate features
        features = torch.cat([
            pos_diff,           # Position difference
            vel_diff,           # Velocity difference
            pos_diff.abs(),     # Absolute position diff
            vel_diff.abs()      # Absolute velocity diff
        ], dim=-1)  # [B, T, N, N, 12]
        
        return features.view(B * T, N, N, 12)
    
    def _compute_adjacency(self, x, threshold: float = 2.0):
        """Compute spatial adjacency matrix for graph attention."""
        if not self.use_graph_attention:
            return None
        
        B, T, N, D = x.shape
        pos = x[:, :, :, :3]
        
        # Compute pairwise distances
        dist = torch.cdist(pos.view(B * T, N, 3), pos.view(B * T, N, 3))
        
        # Adjacency: objects within threshold distance
        adjacency = (dist < threshold).float()
        
        return adjacency.view(B, T, N, N)
    
    def forward(self, x):
        """Forward pass with ablation-aware processing."""
        B, T, N, D = x.shape
        
        # Apply learned feature scaling
        if self.use_learned_scaling:
            x = x * self.feature_scale.view(1, 1, 1, -1)
        
        # Compute physics features for attention bias
        physics_features = self._compute_physics_features(x)
        adjacency = self._compute_adjacency(x)
        
        # Input projection
        h = self.input_proj(x)  # [B, T, N, embed_dim]
        
        # Transformer layers
        for layer in self.layers:
            h = layer(h, physics_features, adjacency)
        
        # Output projection
        h = self.output_norm(h)
        output = self.output_proj(h)
        
        # Energy conservation constraint (if enabled)
        if self.use_energy_conservation and self.training:
            # The energy conservation loss is computed in the training loop
            pass
        
        return output
    
    def compute_energy_loss(self, predictions, targets):
        """Compute energy conservation loss for pHMARL regularization."""
        if not self.use_energy_conservation:
            return torch.tensor(0.0, device=predictions.device)
        
        # Compute kinetic energy: 0.5 * m * v^2
        # Assuming velocity is in dims 3:6
        pred_vel = predictions[:, :, :, 3:6]
        target_vel = targets[:, :, :, 3:6]
        
        pred_ke = 0.5 * (pred_vel ** 2).sum(dim=-1)  # [B, T, N]
        target_ke = 0.5 * (target_vel ** 2).sum(dim=-1)
        
        # Total energy should be conserved across time
        pred_total = pred_ke.sum(dim=-1)  # [B, T]
        target_total = target_ke.sum(dim=-1)
        
        # Energy conservation: variance of total energy over time should be low
        pred_energy_var = pred_total.var(dim=1).mean()
        target_energy_var = target_total.var(dim=1).mean()
        
        # Penalize deviation from target energy profile
        energy_loss = F.mse_loss(pred_total, target_total) + 0.1 * pred_energy_var
        
        return energy_loss


print("✓ PhysicsFormerV2 model defined with ablation support:")
print(f"  - use_physics_bias: Physics-informed attention bias")
print(f"  - use_energy_conservation: pHMARL energy conservation")
print(f"  - use_graph_attention: Body Transformer graph attention")
print(f"  - use_learned_scaling: Per-feature learned scaling")

In [ ]:
# ============================================================
# CELL 5: ABLATION RUNNER WITH REAL TRAINING
# ============================================================

class AblationRunner:
    """Runs ablation experiments with multiple seeds for statistical significance."""
    
    def __init__(self, base_config: AblationConfig, data_path: str, device: str = 'cuda'):
        self.base_config = base_config
        self.data_path = data_path
        self.device = device
        self.results = {}
        
        # Load training data
        self.train_loader, self.val_loader = self._load_data()
        
    def _load_data(self):
        """Load physics training data. Raises error if data not found."""
        data_file = Path(self.data_path) / "physics_sequences.pt"
        if not data_file.exists():
            raise FileNotFoundError(
                f"Training data not found at {data_file}. "
                f"Run data generation first or set correct GDRIVE_DATA path."
            )
        
        print(f"Loading data from {data_file}...")
        data = torch.load(data_file, map_location='cpu')
        
        # Create simple dataset
        class PhysicsDataset(Dataset):
            def __init__(self, sequences, schemas):
                self.sequences = sequences
                self.schemas = schemas
                
            def __len__(self):
                return len(self.sequences)
            
            def __getitem__(self, idx):
                return self.sequences[idx], self.schemas[idx]
        
        # Split data
        n_samples = len(data['sequences'])
        n_train = int(0.9 * n_samples)
        
        train_dataset = PhysicsDataset(
            data['sequences'][:n_train],
            data['schemas'][:n_train]
        )
        val_dataset = PhysicsDataset(
            data['sequences'][n_train:],
            data['schemas'][n_train:]
        )
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        print(f"  Train samples: {len(train_dataset)}")
        print(f"  Val samples: {len(val_dataset)}")
        
        return train_loader, val_loader
    
    def _create_model(self, config: AblationConfig):
        """Create PhysicsFormerV2 model with given config. No fallbacks."""
        model = PhysicsFormerV2(
            num_objects=config.num_objects,
            state_dim=config.state_dim,
            embed_dim=config.embed_dim,
            num_heads=config.num_heads,
            num_layers=config.num_layers,
            ff_dim=config.ff_dim,
            dropout=config.dropout,
            use_physics_bias=config.use_physics_bias,
            use_energy_conservation=config.use_energy_conservation,
            use_graph_attention=config.use_graph_attention,
            use_learned_scaling=config.use_learned_scaling
        )
        return model.to(self.device)
    
    def _train_epoch(self, model, optimizer, criterion):
        """Train for one epoch. Returns average loss."""
        model.train()
        total_loss = 0.0
        n_batches = 0
        
        for sequences, schemas in self.train_loader:
            sequences = sequences.to(self.device)
            schemas = schemas.to(self.device)
            
            optimizer.zero_grad()
            
            # Forward pass - predict next state
            # Input: [B, T-1, N, D], Target: [B, T-1, N, D] (shifted by 1)
            if sequences.shape[1] > 1:
                input_seq = sequences[:, :-1]
                target_seq = sequences[:, 1:]
                
                predictions = model(input_seq)
                loss = criterion(predictions, target_seq)
            else:
                # Single frame - use reconstruction loss
                predictions = model(sequences)
                loss = criterion(predictions, sequences)
            
            # Check for NaN loss
            if torch.isnan(loss):
                raise ValueError("NaN loss encountered during training")
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
        
        return total_loss / n_batches
    
    def _validate(self, model, criterion):
        """Validate model. Returns MSE, accuracy, energy error."""
        model.eval()
        total_mse = 0.0
        correct = 0
        total = 0
        energy_errors = []
        
        with torch.no_grad():
            for sequences, schemas in self.val_loader:
                sequences = sequences.to(self.device)
                schemas = schemas.to(self.device)
                
                # Prediction loss
                if sequences.shape[1] > 1:
                    input_seq = sequences[:, :-1]
                    target_seq = sequences[:, 1:]
                    predictions = model(input_seq)
                    mse = criterion(predictions, target_seq)
                else:
                    predictions = model(sequences)
                    mse = criterion(predictions, sequences)
                
                total_mse += mse.item() * sequences.shape[0]
                
                # Schema classification (if model has classifier)
                if hasattr(model, 'schema_classifier'):
                    # Pool over time and objects
                    pooled = predictions.mean(dim=[1, 2])
                    logits = model.schema_classifier(pooled)
                    pred_schema = logits.argmax(dim=-1)
                    correct += (pred_schema == schemas).sum().item()
                    total += schemas.shape[0]
                
                # Energy conservation check
                if sequences.shape[1] > 1:
                    # Compute kinetic energy: 0.5 * m * v^2
                    # Assuming velocity is in dims 3:6 and mass is dim 6
                    velocities = predictions[:, :, :, 3:6]  # [B, T, N, 3]
                    initial_ke = 0.5 * (velocities[:, 0] ** 2).sum(dim=-1).mean()
                    final_ke = 0.5 * (velocities[:, -1] ** 2).sum(dim=-1).mean()
                    energy_error = abs(final_ke - initial_ke) / (initial_ke + 1e-8)
                    energy_errors.append(energy_error.item())
        
        avg_mse = total_mse / len(self.val_loader.dataset)
        accuracy = (correct / total * 100) if total > 0 else 0.0
        avg_energy_error = np.mean(energy_errors) if energy_errors else 0.0
        
        return avg_mse, accuracy, avg_energy_error
    
    def run_single_experiment(self, config: AblationConfig, seed: int,
                               max_epochs: int = 50, patience: int = 10) -> Dict:
        """Run a single training experiment and return metrics."""
        # Set seeds for reproducibility
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
        
        start_time = time.time()
        
        # Create model
        model = self._create_model(config)
        
        # Setup training
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
        criterion = nn.MSELoss()
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )
        
        # Training loop with early stopping
        best_mse = float('inf')
        best_epoch = 0
        epochs_without_improvement = 0
        
        for epoch in range(max_epochs):
            # Train
            train_loss = self._train_epoch(model, optimizer, criterion)
            
            # Validate
            val_mse, val_acc, energy_error = self._validate(model, criterion)
            
            # Update scheduler
            scheduler.step(val_mse)
            
            # Early stopping check
            if val_mse < best_mse:
                best_mse = val_mse
                best_acc = val_acc
                best_energy_error = energy_error
                best_epoch = epoch + 1
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            
            if epochs_without_improvement >= patience:
                break
        
        training_time = time.time() - start_time
        
        return {
            'seed': seed,
            'final_mse': best_mse,
            'schema_accuracy': best_acc,
            'epochs_to_converge': best_epoch,
            'energy_conservation_error': best_energy_error,
            'training_time': training_time,
            'config': {
                'use_physics_bias': config.use_physics_bias,
                'use_energy_conservation': config.use_energy_conservation,
                'use_graph_attention': config.use_graph_attention,
                'use_learned_scaling': config.use_learned_scaling
            }
        }
    
    def run_ablation(self, condition_name: str, ablation_config: Dict, 
                     num_runs: int = 5, seed_base: int = 42) -> List[Dict]:
        """Run multiple experiments for one ablation condition."""
        config = self.base_config.apply_ablation(ablation_config)
        
        print(f"\n{'─'*60}")
        print(f"  CONDITION: {condition_name}")
        print(f"{'─'*60}")
        print(f"  {ablation_config.get('description', '')}")
        print(f"  physics_bias={config.use_physics_bias}, energy={config.use_energy_conservation}")
        print(f"  graph={config.use_graph_attention}, scaling={config.use_learned_scaling}")
        print(f"  Running {num_runs} experiments (seeds {seed_base}-{seed_base + num_runs - 1})")
        print()
        
        runs = []
        for i in range(num_runs):
            seed = seed_base + i
            print(f"    Run {i+1}/{num_runs} (seed={seed})...", end=" ", flush=True)
            result = self.run_single_experiment(config, seed)
            runs.append(result)
            print(f"MSE={result['final_mse']:.4f}, "
                  f"Acc={result['schema_accuracy']:.1f}%, "
                  f"Epochs={result['epochs_to_converge']}, "
                  f"Time={result['training_time']:.1f}s")
        
        # Show summary for this condition
        mse_vals = [r['final_mse'] for r in runs]
        acc_vals = [r['schema_accuracy'] for r in runs]
        print(f"\n  Summary: MSE={np.mean(mse_vals):.4f}±{np.std(mse_vals):.4f}, "
              f"Acc={np.mean(acc_vals):.1f}±{np.std(acc_vals):.1f}%")
        
        self.results[condition_name] = runs
        return runs
    
    def run_all_ablations(self, conditions: Dict[str, Dict], 
                          num_runs: int = 5, seed_base: int = 42) -> Dict:
        """Run all ablation conditions."""
        total_experiments = len(conditions) * num_runs
        print("\n" + "="*70)
        print("RUNNING ABLATION EXPERIMENTS")
        print("="*70)
        print(f"  Conditions: {len(conditions)}")
        print(f"  Runs per condition: {num_runs}")
        print(f"  Total experiments: {total_experiments}")
        
        start_time = time.time()
        completed = 0
        
        for name, config in conditions.items():
            self.run_ablation(name, config, num_runs, seed_base)
            completed += num_runs
            elapsed = time.time() - start_time
            remaining = (elapsed / completed) * (total_experiments - completed) if completed > 0 else 0
            print(f"\n  Progress: {completed}/{total_experiments} experiments ({100*completed/total_experiments:.0f}%)")
            print(f"  Elapsed: {elapsed/60:.1f}min, Estimated remaining: {remaining/60:.1f}min")
        
        total_time = time.time() - start_time
        print("\n" + "="*70)
        print(f"ALL EXPERIMENTS COMPLETE!")
        print(f"  Total time: {total_time/60:.1f} minutes")
        print("="*70)
        
        return self.results


print("✓ AblationRunner defined with REAL TRAINING")

In [ ]:
# ============================================================
# CELL 5B: GENERATE OR LOAD TRAINING DATA
# ============================================================

def generate_synthetic_physics_data(n_samples: int = 5000, seq_len: int = 32, 
                                     num_objects: int = 10, state_dim: int = 28,
                                     num_schemas: int = 37) -> Dict:
    """Generate synthetic physics sequences for ablation study.
    
    Each sequence simulates basic physics: position updates based on velocity,
    with random initial conditions and simple collision-like interactions.
    """
    print(f"Generating {n_samples} synthetic physics sequences...")
    
    sequences = []
    schemas = []
    
    for i in range(n_samples):
        # Random schema (physics scenario type)
        schema = np.random.randint(0, num_schemas)
        
        # Initialize object states: [pos(3), vel(3), quat(4), ang_vel(3), mass(1), radius(1), props(13)]
        # Simplified: just use position(3), velocity(3), and padding
        states = torch.zeros(seq_len, num_objects, state_dim)
        
        # Random initial positions [-5, 5]
        positions = torch.rand(num_objects, 3) * 10 - 5
        
        # Random initial velocities [-1, 1]
        velocities = torch.rand(num_objects, 3) * 2 - 1
        
        # Random masses [0.5, 2.0]
        masses = torch.rand(num_objects) * 1.5 + 0.5
        
        for t in range(seq_len):
            # Update positions based on velocities (simple Euler integration)
            if t > 0:
                positions = positions + velocities * 0.1  # dt = 0.1
                
                # Simple boundary bounce
                bounce_mask = (positions.abs() > 5).any(dim=1)
                velocities[bounce_mask] *= -0.8  # Damped bounce
                
                # Simple collision check (objects too close)
                for j in range(num_objects):
                    for k in range(j+1, num_objects):
                        dist = (positions[j] - positions[k]).norm()
                        if dist < 0.5:  # Collision threshold
                            # Exchange velocities (simplified)
                            velocities[j], velocities[k] = velocities[k].clone() * 0.9, velocities[j].clone() * 0.9
            
            # Store state
            states[t, :, :3] = positions
            states[t, :, 3:6] = velocities
            states[t, :, 6] = masses
            # Rest of state dimensions are zeros (static properties)
        
        sequences.append(states)
        schemas.append(schema)
        
        if (i + 1) % 1000 == 0:
            print(f"  Generated {i+1}/{n_samples} sequences")
    
    return {
        'sequences': torch.stack(sequences),
        'schemas': torch.tensor(schemas)
    }


# Check if data exists, otherwise generate it
data_file = Path(GDRIVE_DATA) / "physics_sequences.pt"
Path(GDRIVE_DATA).mkdir(parents=True, exist_ok=True)

if data_file.exists():
    print(f"✓ Found existing data at {data_file}")
else:
    print(f"Data not found at {data_file}")
    print("Generating synthetic physics data for ablation study...")
    print()
    
    # Generate data
    data = generate_synthetic_physics_data(
        n_samples=5000,
        seq_len=32,
        num_objects=base_config.num_objects,
        state_dim=base_config.state_dim,
        num_schemas=37
    )
    
    # Save data
    torch.save(data, data_file)
    print(f"\n✓ Saved synthetic data to {data_file}")
    print(f"  Sequences shape: {data['sequences'].shape}")
    print(f"  Schemas shape: {data['schemas'].shape}")

In [ ]:
# ============================================================
# CELL 6: RUN ABLATION EXPERIMENTS
# ============================================================

print("\n" + "="*70)
print("STEP 1: INITIALIZING ABLATION STUDY")
print("="*70)
print(f"  Conditions to test: {len(ABLATION_CONDITIONS)}")
print(f"  Runs per condition: {NUM_RUNS}")
print(f"  Total experiments: {len(ABLATION_CONDITIONS) * NUM_RUNS}")
print()

runner = AblationRunner(base_config, GDRIVE_DATA, device=str(device))
results = runner.run_all_ablations(ABLATION_CONDITIONS, num_runs=NUM_RUNS, seed_base=SEED_BASE)

In [ ]:
# ============================================================
# CELL 7: STATISTICAL ANALYSIS
# ============================================================

print("\n" + "="*70)
print("STEP 2: STATISTICAL ANALYSIS")
print("="*70)

analysis = analyze_ablation_results(results)

print("\n" + "─"*80)
print("RESULTS WITH 95% CONFIDENCE INTERVALS")
print("─"*80)
print(f"\n{'Condition':<25} {'MSE':<30} {'Accuracy (%)':<25} {'Epochs':<20}")
print("─" * 100)

for condition in ABLATION_CONDITIONS.keys():
    a = analysis[condition]
    print(f"{condition:<25} {a['mse']['formatted']:<30} {a['accuracy']['formatted']:<25} {a['convergence']['formatted']:<20}")

print("\n" + "─"*80)
print("STATISTICAL SIGNIFICANCE vs FULL MODEL")
print("─"*80)
print("  Legend: ** p<0.05, *** p<0.01")
print("  Effect size: small (<0.5), medium (0.5-0.8), large (>0.8)")

for condition in ABLATION_CONDITIONS.keys():
    if condition == 'full_model':
        continue
    
    a = analysis[condition]
    vs = a['vs_baseline']
    
    print(f"\n  {condition}:")
    print(f"    {ABLATION_CONDITIONS[condition]['description']}")
    
    # MSE test
    mse_test = vs['mse_test']
    sig = "***" if mse_test['significant_001'] else "**" if mse_test['significant_005'] else ""
    print(f"    MSE: p={mse_test['p_value']:.4f}{sig}, Cohen's d={mse_test['cohens_d']:.2f} ({mse_test['effect_size']})")
    
    # Accuracy test
    acc_test = vs['accuracy_test']
    sig = "***" if acc_test['significant_001'] else "**" if acc_test['significant_005'] else ""
    print(f"    Accuracy: p={acc_test['p_value']:.4f}{sig}, Cohen's d={acc_test['cohens_d']:.2f} ({acc_test['effect_size']})")
    
    # Convergence test
    conv_test = vs['convergence_test']
    sig = "***" if conv_test['significant_001'] else "**" if conv_test['significant_005'] else ""
    print(f"    Convergence: p={conv_test['p_value']:.4f}{sig}, Cohen's d={conv_test['cohens_d']:.2f} ({conv_test['effect_size']})")

In [ ]:
# ============================================================
# CELL 8: FEATURE CONTRIBUTION RANKING
# ============================================================

print("\n" + "="*70)
print("STEP 3: FEATURE CONTRIBUTION RANKING")
print("="*70)
print("\nRanked by impact on MSE (higher = more important):")

# Calculate impact of each feature
baseline_mse = analysis['full_model']['mse']['mean']

feature_impacts = []
for condition, ablation in ABLATION_CONDITIONS.items():
    if condition in ['full_model', 'vanilla_transformer']:
        continue
    
    ablated_mse = analysis[condition]['mse']['mean']
    impact = ablated_mse - baseline_mse
    impact_pct = (impact / baseline_mse) * 100
    
    # Identify which feature was ablated
    feature = condition.replace('no_', '').replace('_', ' ').title()
    
    feature_impacts.append({
        'feature': feature,
        'condition': condition,
        'impact': impact,
        'impact_pct': impact_pct,
        'p_value': analysis[condition]['vs_baseline']['mse_test']['p_value'],
        'significant': analysis[condition]['vs_baseline']['mse_test']['significant_005']
    })

# Sort by impact
feature_impacts.sort(key=lambda x: x['impact'], reverse=True)

print(f"\n{'Rank':<6} {'Feature':<25} {'MSE Impact':<15} {'% Increase':<15} {'p-value':<12} {'Significant'}")
print("─" * 90)

for i, f in enumerate(feature_impacts, 1):
    sig = "Yes **" if f['significant'] else "No"
    print(f"{i:<6} {f['feature']:<25} {f['impact']:+.4f}{'':>7} {f['impact_pct']:+.1f}%{'':>8} {f['p_value']:.4f}{'':>5} {sig}")

# Summary
print("\n" + "="*70)
print("CONCLUSION")
print("="*70)

# Find most important feature
most_important = feature_impacts[0]
print(f"\n  🏆 Most important physics feature: {most_important['feature']}")
print(f"     Removing it increases MSE by {most_important['impact_pct']:.1f}%")
print(f"     Statistical significance: p={most_important['p_value']:.4f}")

# Vanilla transformer comparison
vanilla_mse = analysis['vanilla_transformer']['mse']['mean']
total_impact = vanilla_mse - baseline_mse
total_impact_pct = (total_impact / baseline_mse) * 100

print(f"\n  📊 Total impact of all physics features:")
print(f"     Vanilla transformer MSE: {vanilla_mse:.4f}")
print(f"     Full model MSE: {baseline_mse:.4f}")
print(f"     Combined improvement: {total_impact_pct:.1f}% reduction in MSE")

# Feature importance ranking
print(f"\n  📈 Feature Importance Ranking:")
for i, f in enumerate(feature_impacts, 1):
    sig_marker = "✓" if f['significant'] else "○"
    print(f"     {i}. {f['feature']}: +{f['impact_pct']:.1f}% MSE {sig_marker}")

print("\n" + "="*70)

In [ ]:
# ============================================================
# CELL 9: SAVE RESULTS
# ============================================================

import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
results_file = Path(RESULTS_DIR) / f"ablation_results_{timestamp}.json"

def convert_to_serializable(obj):
    """Convert numpy types to Python native types for JSON serialization."""
    if isinstance(obj, (np.bool_, np.bool)):
        return bool(obj)
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(v) for v in obj]
    return obj

# Prepare results for JSON serialization
save_data = {
    'timestamp': timestamp,
    'config': {
        'num_runs': NUM_RUNS,
        'max_epochs': MAX_EPOCHS,
        'seed_base': SEED_BASE,
        'device': str(device)
    },
    'conditions': ABLATION_CONDITIONS,
    'raw_results': convert_to_serializable(results),
    'analysis': {}
}

# Convert analysis to JSON-serializable format
for condition, a in analysis.items():
    save_data['analysis'][condition] = {
        'mse': {
            'mean': float(a['mse']['mean']),
            'ci_95': [float(a['mse']['ci_95'][0]), float(a['mse']['ci_95'][1])],
            'std': float(a['mse']['std'])
        },
        'accuracy': {
            'mean': float(a['accuracy']['mean']),
            'ci_95': [float(a['accuracy']['ci_95'][0]), float(a['accuracy']['ci_95'][1])],
            'std': float(a['accuracy']['std'])
        },
        'convergence': {
            'mean': float(a['convergence']['mean']),
            'ci_95': [float(a['convergence']['ci_95'][0]), float(a['convergence']['ci_95'][1])],
            'std': float(a['convergence']['std'])
        }
    }
    if 'vs_baseline' in a:
        save_data['analysis'][condition]['vs_baseline'] = {
            'mse_test': convert_to_serializable(a['vs_baseline']['mse_test']),
            'accuracy_test': convert_to_serializable(a['vs_baseline']['accuracy_test']),
            'convergence_test': convert_to_serializable(a['vs_baseline']['convergence_test'])
        }

with open(results_file, 'w') as f:
    json.dump(save_data, f, indent=2)

print(f"✓ Results saved to: {results_file}")
print(f"\nTo load results:")
print(f"  with open('{results_file}', 'r') as f:")
print(f"      data = json.load(f)")